# 92 — Campaign B pre-M6 batch (M4 → M5, stop before M6)

91 で `M3_COMPLETE` になった候補を **M4 → M5** まで進める。

- 経路は notebook 75/76 と同じ（staged child）
- **production M6 は起動しない**（`M6_GATE.json = BLOCKED_PRE_M6`）
- 常に `NOT_CERTIFIED` / `SCREENING_ONLY`
- GPU は 1 本ずつ（global M3/M4 audit を候補ごとに書換）
- M4 未完了なら再実行で resume


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'pre_m6_batch.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/pre_m6_batch.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M4_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 92 requires CUDA GPU for M4.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.pre_m6_batch import list_pre_m6_queue, run_pre_m6_batch

validate_persistent_root()
MAX_PACKAGES = 4            # how many M3_COMPLETE candidates to advance
MAX_STAGE_SESSIONS = 6      # M4 resume loops per package
MAX_QUEUE = 50
ONLY_CAMPAIGN = None

QUEUE = list_pre_m6_queue(
    PERSIST_ROOT,
    max_candidates=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'queue_size': len(QUEUE),
    'top10': [
        {
            'candidate_id': r.get('candidate_id'),
            'q_upper': r.get('q_upper'),
            'stage': r.get('stage'),
            'm3_run_id': r.get('m3_run_id'),
            'm4_run_id': r.get('m4_run_id'),
            'm5_run_id': r.get('m5_run_id'),
        }
        for r in QUEUE[:10]
    ],
}, indent=2, ensure_ascii=False, default=str))


In [ ]:
SESSION = run_pre_m6_batch(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    max_packages=MAX_PACKAGES,
    max_stage_sessions=MAX_STAGE_SESSIONS,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'queue_size': SESSION.get('queue_size'),
    'packages_attempted': SESSION.get('packages_attempted'),
    'pre_m6_ready': SESSION.get('pre_m6_ready'),
    'm4_checkpoint': SESSION.get('m4_checkpoint'),
    'errors': SESSION.get('errors'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_pre_m6' / 'LATEST_PRE_M6_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
print('M6 is NOT started. Check package M6_GATE.json.')
